# MoP + DivPO — Phase 2 & 3 (Kaggle, Maximum GPU)

**Prerequisite:** SFT adapters already at `DasonTio/mop-divpo-coauthor/sft/{persona}/`.

## Kaggle settings (before running)

| Setting | Value |
|---|---|
| Accelerator | **T4 x2** (Settings → Accelerator → GPU T4 x2) |
| Internet | On |
| Secret | `HF_TOKEN` = your HuggingFace write token |

## How to run in background (Save Version)

1. Click **Save Version** (top-right) → **Save & Run All (Commit)** → **Save**
2. Close browser / sleep laptop — job continues on Kaggle servers
3. Kaggle emails when done. Check output at **Your Work → Notebooks**

## Session plan

| Run | Phase | Est. time |
|---|---|---|
| Save Version 1 | Phase 2: DivPO data gen | **~30 min total** |
| Save Version 2 | Phase 3: DivPO training | **~1.5–2h total** |

## GPU optimizations active

| Component | Optimization |
|---|---|
| Generation | SDPA attention, batch=32 prompts × 4 cands = 128 seqs/call, `num_return_sequences` |
| Scoring | Embedder on `cuda:1`, single batch encode per generation batch |
| Training | Accelerate DDP (both T4s), fused AdamW, gradient checkpointing, SDPA, group_by_length |

---
## Cell 1 — Install dependencies

**Save Version:** no restart needed (fresh kernel).
**Interactive:** restart kernel once after this cell, then continue.

In [ ]:
import os
INTERACTIVE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"

!pip install -q --upgrade transformers peft trl accelerate bitsandbytes datasets huggingface_hub sentence-transformers
!pip uninstall -y -q torchao

if INTERACTIVE:
    print("\n>>> INTERACTIVE MODE: restart kernel now, then continue from Cell 2. <<<")
else:
    print("Save Version mode — continuing.")

---
## Cell 2 — Setup

Credentials + repo + working directory + accelerate config for DDP training.
**Run at the start of every session.**

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

REPO_DIR = "/kaggle/working/mop-divpo-llm-counter-argument"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/DasonTio/mop-divpo-llm-counter-argument.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --quiet

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# Write accelerate config for 2-GPU DDP training (Phase 3)
accel_cfg = """\
compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
downcast_bf16: 'no'
gpu_ids: all
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
"""
with open("/tmp/accel_config.yaml", "w") as f:
    f.write(accel_cfg)

import torch
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpu == 0:
    print("NO GPU — go to Notebook settings → Accelerator → T4 x2")
print(f"CWD : {os.getcwd()}")
print(f"HF  : {os.environ['HF_TOKEN'][:8]}...")
print(f"Accelerate config: /tmp/accel_config.yaml ({n_gpu} GPUs)")
print("Ready.")

---
## Phase 2 — DivPO Dataset Generation

- LLM on `cuda:0` with SDPA attention + `cudnn.benchmark`
- Batch: 32 prompts × 4 candidates = 128 sequences per `model.generate()` call
- Embedder on `cuda:1` — single batch encode per generation batch
- Pushes to HF Hub after each persona

**~7–10 min per persona on T4 x2.**

In [ ]:
# Download SFT JSONL prompt pool (skips if already present)
from huggingface_hub import hf_hub_download
import os

os.makedirs("data/processed/sft", exist_ok=True)
for persona in ["contrarian", "systems_thinker", "cross_domain_analogist", "minimalist"]:
    path = f"data/processed/sft/{persona}.jsonl"
    if os.path.exists(path):
        print(f"  {persona}.jsonl already present")
        continue
    hf_hub_download(
        repo_id="DasonTio/mop-divpo-sft-data",
        filename=f"{persona}.jsonl",
        repo_type="dataset",
        local_dir="data/processed/sft",
        token=os.environ["HF_TOKEN"],
    )
    print(f"  Downloaded {persona}.jsonl")

In [ ]:
# DivPO pairs — contrarian
# 32 prompts × 4 candidates = 128 sequences per generate call (~8.5GB VRAM, T4 has 15GB)
!python scripts/prepare_divpo_datasets.py --persona contrarian --from-hub --candidate-count 4 --gen-batch-size 32 --push

In [ ]:
# DivPO pairs — systems_thinker
!python scripts/prepare_divpo_datasets.py --persona systems_thinker --from-hub --candidate-count 4 --gen-batch-size 32 --push

In [ ]:
# DivPO pairs — cross_domain_analogist
!python scripts/prepare_divpo_datasets.py --persona cross_domain_analogist --from-hub --candidate-count 4 --gen-batch-size 32 --push

In [ ]:
# DivPO pairs — minimalist
!python scripts/prepare_divpo_datasets.py --persona minimalist --from-hub --candidate-count 4 --gen-batch-size 32 --push

In [ ]:
# Verify all 4 persona files on HF Hub
from huggingface_hub import list_repo_files
import os

files = sorted(list_repo_files("DasonTio/mop-divpo-divpo-data", repo_type="dataset", token=os.environ["HF_TOKEN"]))
print("DivPO data on Hub:")
for f in files:
    print(" ", f)
expected = {"contrarian.jsonl", "systems_thinker.jsonl", "cross_domain_analogist.jsonl", "minimalist.jsonl"}
missing = expected - set(files)
print("\nMissing:", missing if missing else "none — Phase 2 complete")

---
## Phase 3 — DivPO Training

Uses `accelerate launch` for true 2-GPU DDP training:
- Process 0 on `cuda:0`, Process 1 on `cuda:1`
- Each process: trainable model + ref model on its own GPU (no cross-device issues)
- Effective batch = 8 per device × 2 GPUs × 4 grad_accum = 64
- SDPA attention, fused AdamW, gradient checkpointing, group_by_length

**~20–25 min per persona on T4 x2 (was ~60 min single GPU).**

> Starting a new Save Version for Phase 3? Cell 2 runs automatically first.

In [ ]:
# DivPO training — contrarian (2-GPU DDP via accelerate)
!ACCELERATE_CONFIG_FILE=/tmp/accel_config.yaml accelerate launch scripts/train_divpo.py --persona contrarian

In [ ]:
# DivPO training — systems_thinker
!ACCELERATE_CONFIG_FILE=/tmp/accel_config.yaml accelerate launch scripts/train_divpo.py --persona systems_thinker

In [ ]:
# DivPO training — cross_domain_analogist
!ACCELERATE_CONFIG_FILE=/tmp/accel_config.yaml accelerate launch scripts/train_divpo.py --persona cross_domain_analogist

In [ ]:
# DivPO training — minimalist
!ACCELERATE_CONFIG_FILE=/tmp/accel_config.yaml accelerate launch scripts/train_divpo.py --persona minimalist

---
## Verify — Load DivPO adapter and generate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")

# Change subfolder to test other personas: divpo/systems_thinker, divpo/minimalist, etc.
model = PeftModel.from_pretrained(base, "DasonTio/mop-divpo-coauthor", subfolder="divpo/contrarian")
model.eval()

prompt = "Generate a counter-argument to this claim:\n\nRemote work is strictly better for productivity."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.9, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))